# Submission 1 — Full Champion Pipeline

**Objective:** Train all 6 base models from scratch + stacking ensemble → final submission.

**Champion:** S7 — Stacking of C_focal_v1 + C_focal_v2 + ConvNeXt + ConvNeXtV2 + EffNetB3 + ResNet50

| Model | Arch | Loss | Epochs | Strategy |
|:---|---:|:---|---:|:---|
| C_focal_v1 | CLIP ViT-B/32 + Text Features | FocalLoss(γ=2, α=class_weight) | 10 | Head-only |
| C_focal_v2 | CLIP ViT-B/32 + Text Features | CrossEntropy(label_smooth=0.1) | 20 | Head-only |
| ConvNeXt | convnext_tiny | CrossEntropy(class_weight) | 10 head + 20 fine-tune | Two-phase |
| ConvNeXtV2 | convnextv2_tiny | CrossEntropy | 15 | Head-only |
| EffNet B3 | efficientnet_b3 @ 300px | CrossEntropy(class_weight) | 10 head + 20 fine-tune | Two-phase |
| ResNet50 | resnet50 | CrossEntropy(class_weight) | 10 head + 20 fine-tune | Two-phase |

**Meta-learner:** LogisticRegression(C=1, max_iter=1000, solver='lbfgs', random_state=42)

**Expected Val F1:** ~0.9872

In [1]:
import sys, json, gc, warnings
sys.path.insert(0, "../..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression

import open_clip

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.utils.load_data import load_train, load_test
from modules.models.factory import TrashClassifier
from modules.data.dataset import get_transforms
from modules.training.train import fit

warnings.filterwarnings('ignore')
set_seed(SEED)

print(f"Device: {DEVICE}")
print(f"open_clip: {open_clip.__version__}")

/home/prayudahlah/projects/external/big-data-big-trouble/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
open_clip: 3.3.0


---
## Data Preparation

We use a single stratified 80/20 split (seed=42) shared by all models.
Multiple dataloader variants are created to handle different input transforms:
1. **CLIP native transform** (224px) — for C_focal models
2. **CNN 224px** — for ConvNeXt, ConvNeXtV2, ResNet50
3. **CNN 300px** — for EfficientNet B3

In [2]:
CLASS_LABELS_LIST = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_TO_IDX = {l: i for i, l in enumerate(CLASS_LABELS_LIST)}
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]

df_all = load_train()
y_all = df_all["label"].map(CLASS_TO_IDX).values

from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(df_all, y_all))
df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_val = df_all.iloc[val_idx].reset_index(drop=True)

counts = np.bincount([CLASS_TO_IDX[l] for l in df_train["label"]], minlength=3)
class_weights = torch.FloatTensor(counts.sum() / (3 * counts))

print(f"Train: {len(df_train)} | Val: {len(df_val)}")
print(f"Class weights: {class_weights.tolist()}")
print(f"Train distribution: {dict(zip(CLASS_LABELS_LIST, counts))}")

Train: 21221 | Val: 5306
Class weights: [0.8843188881874084, 2.232144832611084, 0.7036373615264893]
Train distribution: {'0_Recyclable': np.int64(7999), '1_Electronic': np.int64(3169), '2_Organic': np.int64(10053)}


In [3]:
class SingleTransformDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), CLASS_TO_IDX[row['label']]


conv_train_tfm = get_transforms(augment=True,  img_size=224)
conv_val_tfm   = get_transforms(augment=False, img_size=224)
conv_train_tfm_300 = get_transforms(augment=True,  img_size=300)
conv_val_tfm_300   = get_transforms(augment=False, img_size=300)

conv_train_224 = DataLoader(
    SingleTransformDataset(df_train, conv_train_tfm),
    batch_size=32, shuffle=True,  num_workers=4, pin_memory=True,
)
conv_val_224 = DataLoader(
    SingleTransformDataset(df_val, conv_val_tfm),
    batch_size=32, shuffle=False, num_workers=4, pin_memory=True,
)

conv_train_300 = DataLoader(
    SingleTransformDataset(df_train, conv_train_tfm_300),
    batch_size=16, shuffle=True,  num_workers=4, pin_memory=True,
)
conv_val_300 = DataLoader(
    SingleTransformDataset(df_val, conv_val_tfm_300),
    batch_size=16, shuffle=False, num_workers=4, pin_memory=True,
)

print("Dataloaders ready:")
print(f"  CNN 224px: train {len(conv_train_224)} batches, val {len(conv_val_224)} batches")
print(f"  CNN 300px: train {len(conv_train_300)} batches, val {len(conv_val_300)} batches")

Dataloaders ready:
  CNN 224px: train 664 batches, val 166 batches
  CNN 300px: train 1327 batches, val 332 batches


---
## CLIP Backbone — ViT-B/32 (OpenCLIP)

We use OpenCLIP's ViT-B/32 pretrained on LAION-2B.
The CLIP visual encoder is frozen for all experiments (head-only training).
Text features are computed from class-specific prompts and concatenated with visual features.

In [4]:
CLIP_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'
clip_model, _, clip_transform = open_clip.create_model_and_transforms(
    CLIP_NAME, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_NAME)
clip_model = clip_model.to('cpu')
clip_model.eval()
clip_visual = clip_model.visual.to(DEVICE)
clip_dim = clip_visual.output_dim

print(f"CLIP dim: {clip_dim}")

prompts = [
    "a photo of recyclable items like paper, plastic, glass, and metal",
    "a photo of electronic waste like computers, phones, cables, and batteries",
    "a photo of organic waste like food scraps, leaves, and plants",
]

# CLIP dataloaders
clip_train_loader = DataLoader(
    SingleTransformDataset(df_train, clip_transform),
    batch_size=32, shuffle=True,  num_workers=4, pin_memory=True,
)
clip_val_loader = DataLoader(
    SingleTransformDataset(df_val, clip_transform),
    batch_size=32, shuffle=False, num_workers=4, pin_memory=True,
)

print(f"CLIP loaders: train {len(clip_train_loader)} batches, val {len(clip_val_loader)} batches")

CLIP dim: 512
CLIP loaders: train 664 batches, val 166 batches


---
## Model Definitions — CLIPWithTextFeatures, FocalLoss, Helpers

Key architectural decisions:
- **CLIPWithTextFeatures:** Concatenates visual features (512-d) with text similarity scores (3-d) before the classification head. This lets the model leverage CLIP's zero-shot knowledge via text prompts while learning task-specific patterns.
- **FocalLoss:** Down-weights easy examples and focuses on hard ones. Gamma=2 is standard; alpha uses inverse class frequency to handle imbalance.
- **train_head_only:** Trains only the classifier head with frozen encoder. Uses AdamW(lr=1e-3, wd=1e-4) + CosineAnnealingLR.
- **get_probs:** Efficient inference helper that returns softmax probabilities.

In [5]:
class CLIPWithTextFeatures(nn.Module):
    """CLIP visual encoder + text similarity features + classifier head."""
    def __init__(self, clip_model, clip_visual, prompts, num_classes=3):
        super().__init__()
        self.visual = clip_visual
        self.encoder = self.visual
        self.logit_scale = clip_model.logit_scale
        dim = self.visual.output_dim
        text_tokens = clip_tokenizer(prompts)
        with torch.no_grad():
            text_feat = clip_model.encode_text(text_tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        self.register_buffer('text_features', text_feat)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + len(prompts), 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        v = self.visual(x)
        v_norm = v / v.norm(dim=-1, keepdim=True)
        sim = v_norm @ self.text_features.T
        combined = torch.cat([v, sim * self.logit_scale.exp()], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = True


class FocalLoss(nn.Module):
    """Focal Loss: down-weights easy examples, focuses on hard/misclassified."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        log_sm = F.log_softmax(inputs, dim=1)
        log_pt = log_sm[range(len(targets)), targets]
        pt = log_pt.exp()
        focal = -((1 - pt).clamp(min=1e-8).pow(self.gamma)) * log_pt
        if self.alpha is not None:
            focal = self.alpha[targets] * focal
        return focal.mean()


def get_probs(model, loader):
    """Run model on loader and return softmax probabilities (numpy)."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc="Inference", leave=False):
            x = x.to(DEVICE)
            all_probs.append(torch.softmax(model(x), dim=1).cpu().numpy())
    return np.concatenate(all_probs)


def train_head_only(model, train_loader, val_loader, name, epochs=10, criterion=None):
    """Train classifier head only with frozen encoder. Saves best checkpoint by val F1."""
    model = model.to(DEVICE)
    model.freeze_encoder()
    criterion = criterion or nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        for x, y in tqdm(train_loader, desc=f"  E{epoch+1}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
        sched.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"  E{epoch+1:02d}: val_f1={f1:.4f} (best={best_f1:.4f})")
    model.load_state_dict(best_state)
    result = {'name': name, 'best_val_f1': best_f1}
    torch.save(best_state, RESULTS / f'{name}.pt')
    json.dump(result, open(RESULTS / f'{name}.json', 'w'))
    return model, best_f1


all_val_probs = {}  # populated after each model is trained

---
## Phase 1: C_focal_v1 — CLIP + Focal Loss

Trains the CLIP classifier head with Focal Loss (gamma=2.0) for 10 epochs.
Focal Loss addresses class imbalance by down-weighting well-classified examples.
Alpha weights are set to inverse class frequency so the minority class (Electronic) receives higher gradient contribution.

In [6]:
print("=" * 60)
print("Phase 1: C_focal_v1 — Focal Loss (10 epochs)")
print("=" * 60)

focal_loss = FocalLoss(alpha=class_weights.to(DEVICE), gamma=2.0)

m_cv1 = CLIPWithTextFeatures(clip_model, clip_visual, prompts)
m_cv1, f1_cv1 = train_head_only(
    m_cv1, clip_train_loader, clip_val_loader,
    'submission_C_focal_v1', epochs=10, criterion=focal_loss,
)
print(f"C_focal_v1 (Focal Loss γ=2): {f1_cv1:.4f}")

all_val_probs['C_focal_v1'] = get_probs(m_cv1, clip_val_loader)
del m_cv1; torch.cuda.empty_cache(); gc.collect()

Phase 1: C_focal_v1 — Focal Loss (10 epochs)


  E01: val_f1=0.9686 (best=0.9686)


  E02: val_f1=0.9760 (best=0.9760)


  E03: val_f1=0.9776 (best=0.9776)


  E04: val_f1=0.9791 (best=0.9791)


...

  E07: val_f1=0.9816 (best=0.9816)


  E08: val_f1=0.9815 (best=0.9816)


  E09: val_f1=0.9823 (best=0.9823)


  E10: val_f1=0.9825 (best=0.9825)
C_focal_v1 (Focal Loss γ=2): 0.9825


9

---
## Phase 2: C_focal_v2 — CLIP + Label Smoothing (20 epochs)

Same CLIP architecture but trained with Label Smoothing (ε=0.1) for 20 epochs.
Label smoothing prevents the model from being overconfident by softening target distributions.
The longer training schedule (20 vs 10 epochs) with smoothing often yields better calibrated probabilities for ensemble stacking.

In [7]:
print("=" * 60)
print("Phase 2: C_focal_v2 — Label Smoothing (20 epochs)")
print("=" * 60)

criterion_ls = nn.CrossEntropyLoss(label_smoothing=0.1)

m_cv2 = CLIPWithTextFeatures(clip_model, clip_visual, prompts)
m_cv2, f1_cv2 = train_head_only(
    m_cv2, clip_train_loader, clip_val_loader,
    'submission_C_focal_v2', epochs=20, criterion=criterion_ls,
)
print(f"C_focal_v2 (Label Smoothing 0.1, 20ep): {f1_cv2:.4f}")

all_val_probs['C_focal_v2'] = get_probs(m_cv2, clip_val_loader)
del m_cv2; torch.cuda.empty_cache(); gc.collect()

Phase 2: C_focal_v2 — Label Smoothing (20 epochs)


  E01: val_f1=0.9741 (best=0.9741)


  E02: val_f1=0.9779 (best=0.9779)


  E03: val_f1=0.9814 (best=0.9814)


  E04: val_f1=0.9820 (best=0.9820)


...

  E17: val_f1=0.9844 (best=0.9861)


  E18: val_f1=0.9847 (best=0.9861)


  E19: val_f1=0.9844 (best=0.9861)


  E20: val_f1=0.9847 (best=0.9861)
C_focal_v2 (Label Smoothing 0.1, 20ep): 0.9861


9

---
## Phase 3: ConvNeXt Tiny — Two-Phase Training

Uses the shared `fit()` pipeline from `modules.training.train`:
1. **Phase 1 (10 epochs):** Head-only — train classifier, freeze encoder
2. **Phase 2 (20 epochs):** Full fine-tune — encoder at lr=1e-4, classifier at lr=1e-3

Mixed precision (AMP) is used for faster training. Early stopping with patience=10 prevents overfitting.
Class weights handle the imbalanced dataset. However, the best checkpoint typically comes from Phase 1.

In [8]:
print("=" * 60)
print("Phase 3: ConvNeXt Tiny — Two-Phase")
print("=" * 60)

model_conv = TrashClassifier('convnext_tiny', num_classes=3)
result_conv = fit(
    model_conv, conv_train_224, conv_val_224,
    name='submission_convnext_tiny',
    encoder_name='convnext_tiny',
    epochs_head=10, epochs_finetune=20,
    class_weights=class_weights,
)
print(f"ConvNeXt Tiny: {result_conv['best_val_f1']:.4f}")

model_conv = TrashClassifier('convnext_tiny', num_classes=3)
model_conv.load_state_dict(torch.load(RESULTS / 'submission_convnext_tiny.pt', map_location=DEVICE, weights_only=True))
model_conv = model_conv.to(DEVICE)
all_val_probs['ConvNeXt'] = get_probs(model_conv, conv_val_224)
del model_conv; torch.cuda.empty_cache(); gc.collect()

Phase 3: ConvNeXt Tiny — Two-Phase

=== submission_convnext_tiny: Phase 1 — Head Only ===


  E01: train_loss=0.1851  val_f1=0.9587  best=0.9587


  E02: train_loss=0.1269  val_f1=0.9642  best=0.9642


  E03: train_loss=0.1039  val_f1=0.9705  best=0.9705


  E04: train_loss=0.0895  val_f1=0.9671  best=0.9705


...

  E18: train_loss=0.0446  val_f1=0.9572  best=0.9729


  E19: train_loss=0.0332  val_f1=0.9575  best=0.9729


  E20: train_loss=0.0290  val_f1=0.9564  best=0.9729
  Early stopping at epoch 20


ConvNeXt Tiny: 0.9729


8

---
## Phase 4: ConvNeXt V2 Tiny — Head-Only (15 epochs)

ConvNeXt V2 uses an improved convolutional design with fully convolutional masked autoencoder pretraining.
Trained head-only for 15 epochs (standard CrossEntropyLoss, no class weights).
This provides diversity from ConvNeXt V1 in the ensemble.

In [9]:
print("=" * 60)
print("Phase 4: ConvNeXt V2 Tiny — Head-Only (15 epochs)")
print("=" * 60)

m_conv_v2 = TrashClassifier('convnextv2_tiny', num_classes=3)
m_conv_v2, f1_conv_v2 = train_head_only(
    m_conv_v2, conv_train_224, conv_val_224,
    'submission_convnextv2_tiny', epochs=15,
)
print(f"ConvNeXt V2: {f1_conv_v2:.4f}")

all_val_probs['ConvNeXtV2'] = get_probs(m_conv_v2, conv_val_224)
del m_conv_v2; torch.cuda.empty_cache(); gc.collect()

Phase 4: ConvNeXt V2 Tiny — Head-Only (15 epochs)


  E01: val_f1=0.9634 (best=0.9634)


  E02: val_f1=0.9668 (best=0.9668)


  E03: val_f1=0.9696 (best=0.9696)


  E04: val_f1=0.9672 (best=0.9696)


...

  E12: val_f1=0.9738 (best=0.9738)


  E13: val_f1=0.9729 (best=0.9738)


  E14: val_f1=0.9736 (best=0.9738)


  E15: val_f1=0.9739 (best=0.9739)
ConvNeXt V2: 0.9739


0

---
## Phase 5: EfficientNet B3 @ 300px — Two-Phase Training

EfficientNet B3 uses neural architecture search to find optimal depth/width/resolution scaling.
Trained at 300px resolution (vs 224px for other models) for richer visual detail.
Batch size reduced to 16 due to larger input size. Two-phase training with early stopping.

In [10]:
print("=" * 60)
print("Phase 5: EfficientNet B3 @ 300px — Two-Phase")
print("=" * 60)

model_eff = TrashClassifier('efficientnet_b3', num_classes=3)
result_eff = fit(
    model_eff, conv_train_300, conv_val_300,
    name='submission_effnet_b3_300',
    encoder_name='efficientnet_b3',
    epochs_head=10, epochs_finetune=20,
    class_weights=class_weights,
)
print(f"EffNet B3 300px: {result_eff['best_val_f1']:.4f}")

model_eff = TrashClassifier('efficientnet_b3', num_classes=3)
model_eff.load_state_dict(torch.load(RESULTS / 'submission_effnet_b3_300.pt', map_location=DEVICE, weights_only=True))
model_eff = model_eff.to(DEVICE)
all_val_probs['EffNet'] = get_probs(model_eff, conv_val_300)
del model_eff; torch.cuda.empty_cache(); gc.collect()

Phase 5: EfficientNet B3 @ 300px — Two-Phase

=== submission_effnet_b3_300: Phase 1 — Head Only ===


  E01: train_loss=0.3303  val_f1=0.9109  best=0.9109


  E02: train_loss=0.2586  val_f1=0.9127  best=0.9127


  E03: train_loss=0.2272  val_f1=0.9192  best=0.9192


  E04: train_loss=0.2087  val_f1=0.9263  best=0.9263


...

  E28: train_loss=0.0032  val_f1=0.9731  best=0.9738


  E29: train_loss=0.0031  val_f1=0.9739  best=0.9739


  E30: train_loss=0.0026  val_f1=0.9733  best=0.9739


EffNet B3 300px: 0.9739


8

---
## Phase 6: ResNet50 — Two-Phase Training

ResNet50 provides a strong convolutional baseline with residual connections.
Two-phase training (10 head + 20 fine-tune) with AMP and class weights.

In [11]:
print("=" * 60)
print("Phase 6: ResNet50 — Two-Phase")
print("=" * 60)

model_rn = TrashClassifier('resnet50', num_classes=3)
result_rn = fit(
    model_rn, conv_train_224, conv_val_224,
    name='submission_resnet50',
    encoder_name='resnet50',
    epochs_head=10, epochs_finetune=20,
    class_weights=class_weights,
)
print(f"ResNet50: {result_rn['best_val_f1']:.4f}")

model_rn = TrashClassifier('resnet50', num_classes=3)
model_rn.load_state_dict(torch.load(RESULTS / 'submission_resnet50.pt', map_location=DEVICE, weights_only=True))
model_rn = model_rn.to(DEVICE)
all_val_probs['ResNet50'] = get_probs(model_rn, conv_val_224)
del model_rn; torch.cuda.empty_cache(); gc.collect()

Phase 6: ResNet50 — Two-Phase

=== submission_resnet50: Phase 1 — Head Only ===


  E01: train_loss=0.4152  val_f1=0.8965  best=0.8965


  E02: train_loss=0.3254  val_f1=0.9013  best=0.9013


  E03: train_loss=0.3112  val_f1=0.9073  best=0.9073


  E04: train_loss=0.2965  val_f1=0.9136  best=0.9136


...

  E28: train_loss=0.0225  val_f1=0.9671  best=0.9684


  E29: train_loss=0.0205  val_f1=0.9660  best=0.9684


  E30: train_loss=0.0198  val_f1=0.9701  best=0.9701


ResNet50: 0.9701


17

---
## Phase 7: Collect Validation Probabilities

All 6 model probabilities have been collected during training.
We now verify the individual model F1 scores and prepare the stacking feature matrix.

In [12]:
val_labels = np.array([CLASS_TO_IDX[r['label']] for _, r in df_val.iterrows()])
print(f"Validation labels: {len(val_labels)}")

print("\nIndividual model F1 scores:")
for name, probs in all_val_probs.items():
    preds = probs.argmax(axis=1)
    f1 = f1_score(val_labels, preds, average='macro')
    f1_per = f1_score(val_labels, preds, average=None)
    print(f"  {name:15s} F1={f1:.4f}  [{f1_per[0]:.4f} {f1_per[1]:.4f} {f1_per[2]:.4f}]")

Validation labels: 5306

Individual model F1 scores:
  C_focal_v1      F1=0.9825  [0.9744 0.9931 0.9800]
  C_focal_v2      F1=0.9847  [0.9776 0.9937 0.9829]
  ConvNeXt        F1=0.9564  [0.9418 0.9675 0.9597]
  ConvNeXtV2      F1=0.9739  [0.9642 0.9842 0.9735]
  EffNet          F1=0.9733  [0.9619 0.9867 0.9713]
  ResNet50        F1=0.9698  [0.9565 0.9855 0.9673]


---
## Phase 8: Stacking Ensemble — LogisticRegression

Instead of fixed weights, we learn optimal per-class per-model weights via LogisticRegression.
Each model contributes 3 probability outputs → 18 features total (6 × 3).
The LR assigns weights per class, allowing different models to specialize on different categories.

**Why LR works:** The softmax outputs are already well-calibrated probabilities. A linear model on top is sufficient to learn optimal combination weights without overfitting.

In [13]:
print("=" * 60)
print("Phase 8: Stacking Ensemble")
print("=" * 60)

model_order = ['C_focal_v1', 'ConvNeXt', 'EffNet', 'ResNet50', 'C_focal_v2', 'ConvNeXtV2']
X_val = np.concatenate([all_val_probs[k] for k in model_order], axis=1)
print(f"Stacking feature matrix: {X_val.shape}")

lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
lr.fit(X_val, val_labels)
preds_val = lr.predict(X_val)

f1_stack = f1_score(val_labels, preds_val, average='macro')
f1_stack_per = f1_score(val_labels, preds_val, average=None)

print(f"\nChampion S7 — All 6 models: F1={f1_stack:.4f}")
print(f"  Per-class: Recyclable={f1_stack_per[0]:.4f}  Electronic={f1_stack_per[1]:.4f}  Organic={f1_stack_per[2]:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(val_labels, preds_val)
print(pd.DataFrame(cm,
    index=[f"True {l}" for l in CLASS_LABELS_LIST],
    columns=[f"Pred {l}" for l in CLASS_LABELS_LIST]))

print("\nClassification Report:")
print(classification_report(val_labels, preds_val, target_names=CLASS_LABELS_LIST, digits=4))

Phase 8: Stacking Ensemble
Stacking feature matrix: (5306, 18)

Champion S7 — All 6 models: F1=0.9874
  Per-class: Recyclable=0.9813  Electronic=0.9956  Organic=0.9853

Confusion Matrix:
                   Pred 0_Recyclable  Pred 1_Electronic  Pred 2_Organic
True 0_Recyclable               1967                  1              32
True 1_Electronic                  3                787               2
True 2_Organic                    39                  1            2474

Classification Report:
              precision    recall  f1-score   support

0_Recyclable     0.9791    0.9835    0.9813      2000
1_Electronic     0.9975    0.9937    0.9956       792
   2_Organic     0.9864    0.9841    0.9853      2514

    accuracy                         0.9853      5306
   macro avg     0.9877    0.9871    0.9874      5306
weighted avg     0.9853    0.9853    0.9853      5306



---
## Phase 9: Test Inference — Generate Submission

Run all 6 models on the test set (1458 images), collect probabilities,
apply the trained LogisticRegression meta-model, and save submission CSV.

**Submission format:** `id` (int), `predicted` (int: 0=Recyclable, 1=Electronic, 2=Organic)

In [14]:
test_df = load_test()
print(f"Test images: {len(test_df)}")

class TestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row["path"]).convert("RGB")
        return self.transform(img), int(row["image_id"])

test_clip_loader = DataLoader(
    TestDataset(test_df, clip_transform),
    batch_size=32, shuffle=False, num_workers=4, pin_memory=True,
)
test_conv_loader = DataLoader(
    TestDataset(test_df, conv_val_tfm),
    batch_size=32, shuffle=False, num_workers=4, pin_memory=True,
)
test_conv_loader_300 = DataLoader(
    TestDataset(test_df, conv_val_tfm_300),
    batch_size=16, shuffle=False, num_workers=4, pin_memory=True,
)
print(f"Test loaders: CLIP={len(test_clip_loader)}, CNN={len(test_conv_loader)}, CNN300={len(test_conv_loader_300)}")

Test images: 1458
Test loaders: CLIP=46, CNN=46, CNN300=92


In [15]:
print("=" * 60)
print("Test Inference — All 6 Models")
print("=" * 60)

# Reconstruct CLIP models for test inference
m_cv1_test = CLIPWithTextFeatures(clip_model, clip_visual, prompts).to(DEVICE)
m_cv1_test.load_state_dict(torch.load(RESULTS / 'submission_C_focal_v1.pt', map_location=DEVICE, weights_only=True))
m_cv1_test.eval()
test_probs_cv1 = get_probs(m_cv1_test, test_clip_loader)
del m_cv1_test; torch.cuda.empty_cache(); gc.collect()

m_cv2_test = CLIPWithTextFeatures(clip_model, clip_visual, prompts).to(DEVICE)
m_cv2_test.load_state_dict(torch.load(RESULTS / 'submission_C_focal_v2.pt', map_location=DEVICE, weights_only=True))
m_cv2_test.eval()
test_probs_cv2 = get_probs(m_cv2_test, test_clip_loader)
del m_cv2_test; torch.cuda.empty_cache(); gc.collect()

# CNN models
m_conv_test = TrashClassifier('convnext_tiny', num_classes=3).to(DEVICE)
m_conv_test.load_state_dict(torch.load(RESULTS / 'submission_convnext_tiny.pt', map_location=DEVICE, weights_only=True))
m_conv_test.eval()
test_probs_conv = get_probs(m_conv_test, test_conv_loader)
del m_conv_test; torch.cuda.empty_cache(); gc.collect()

m_conv_v2_test = TrashClassifier('convnextv2_tiny', num_classes=3).to(DEVICE)
m_conv_v2_test.load_state_dict(torch.load(RESULTS / 'submission_convnextv2_tiny.pt', map_location=DEVICE, weights_only=True))
m_conv_v2_test.eval()
test_probs_conv_v2 = get_probs(m_conv_v2_test, test_conv_loader)
del m_conv_v2_test; torch.cuda.empty_cache(); gc.collect()

m_eff_test = TrashClassifier('efficientnet_b3', num_classes=3).to(DEVICE)
m_eff_test.load_state_dict(torch.load(RESULTS / 'submission_effnet_b3_300.pt', map_location=DEVICE, weights_only=True))
m_eff_test.eval()
test_probs_eff = get_probs(m_eff_test, test_conv_loader_300)
del m_eff_test; torch.cuda.empty_cache(); gc.collect()

m_rn_test = TrashClassifier('resnet50', num_classes=3).to(DEVICE)
m_rn_test.load_state_dict(torch.load(RESULTS / 'submission_resnet50.pt', map_location=DEVICE, weights_only=True))
m_rn_test.eval()
test_probs_rn = get_probs(m_rn_test, test_conv_loader)
del m_rn_test; torch.cuda.empty_cache(); gc.collect()

print("\nAll test probabilities collected.")
print(f"  C_focal_v1: {test_probs_cv1.shape}")
print(f"  C_focal_v2: {test_probs_cv2.shape}")
print(f"  ConvNeXt:   {test_probs_conv.shape}")
print(f"  ConvNeXtV2: {test_probs_conv_v2.shape}")
print(f"  EffNet:     {test_probs_eff.shape}")
print(f"  ResNet50:   {test_probs_rn.shape}")

Test Inference — All 6 Models



All test probabilities collected.
  C_focal_v1: (1458, 3)
  C_focal_v2: (1458, 3)
  ConvNeXt:   (1458, 3)
  ConvNeXtV2: (1458, 3)
  EffNet:     (1458, 3)
  ResNet50:   (1458, 3)


In [16]:
# Build test feature matrix and predict
X_test = np.concatenate([
    test_probs_cv1, test_probs_conv, test_probs_eff,
    test_probs_rn, test_probs_cv2, test_probs_conv_v2,
], axis=1)

test_preds = lr.predict(X_test)

# Save submission
submission_ids = test_df["image_id"].astype(int).values
df_sub = pd.DataFrame({"id": submission_ids, "predicted": test_preds})
df_sub = df_sub.sort_values("id").reset_index(drop=True)

OUTPUT_CSV = PROJECT_ROOT / "submission_1.csv"
df_sub.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")

print("\nSubmission preview:")
print(df_sub.head(10))

print("\nPrediction distribution:")
label_map = {0: 'Recyclable', 1: 'Electronic', 2: 'Organic'}
dist = df_sub['predicted'].map(label_map).value_counts()
print(dist.to_string())

Saved: /home/prayudahlah/projects/external/big-data-big-trouble/submission_1.csv

Submission preview:
   id  predicted
0   1          2
1   2          2
2   3          2
3   4          1
4   5          0
5   6          2
6   7          2
7   8          0
8   9          1
9  10          2

Prediction distribution:
predicted
Organic       738
Recyclable    520
Electronic    200


---
## Final Summary

In [17]:
print("=" * 70)
print("FINAL SUMMARY — Submission 1")
print("=" * 70)

print("\nIndividual model performance (val set):")
print(f"  {'Model':20s} {'F1':8s} {'Recyclable':12s} {'Electronic':12s} {'Organic':12s}")
print("  " + "-" * 64)
for name, probs in all_val_probs.items():
    preds = probs.argmax(axis=1)
    f1 = f1_score(val_labels, preds, average='macro')
    f1_per = f1_score(val_labels, preds, average=None)
    print(f"  {name:20s} {f1:.4f}    {f1_per[0]:.4f}       {f1_per[1]:.4f}       {f1_per[2]:.4f}")

print(f"\n{'=' * 70}")
print(f"🏆 Champion Stacking (All 6 models): F1 = {f1_stack:.4f}")
print(f"{'=' * 70}")
print(f"\nPer-class F1:")
print(f"  Recyclable (0): {f1_stack_per[0]:.4f}")
print(f"  Electronic (1): {f1_stack_per[1]:.4f}")
print(f"  Organic    (2): {f1_stack_per[2]:.4f}")

print(f"\nSubmission saved to: {OUTPUT_CSV}")
print(f"Test predictions: {len(test_preds)} images")
print(f"Distribution: {pd.Series(test_preds).value_counts().to_dict()}")

FINAL SUMMARY — Submission 1

Individual model performance (val set):
  Model                F1       Recyclable   Electronic   Organic     
  ----------------------------------------------------------------
  C_focal_v1           0.9825    0.9744       0.9931       0.9800
  C_focal_v2           0.9847    0.9776       0.9937       0.9829
  ConvNeXt             0.9564    0.9418       0.9675       0.9597
  ConvNeXtV2           0.9739    0.9642       0.9842       0.9735
  EffNet               0.9733    0.9619       0.9867       0.9713
  ResNet50             0.9698    0.9565       0.9855       0.9673

🏆 Champion Stacking (All 6 models): F1 = 0.9874

Per-class F1:
  Recyclable (0): 0.9813
  Electronic (1): 0.9956
  Organic    (2): 0.9853

Submission saved to: /home/prayudahlah/projects/external/big-data-big-trouble/submission_1.csv
Test predictions: 1458 images
Distribution: {2: 738, 0: 520, 1: 200}
